In [9]:
pip install jpype1

Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd, numpy as np, re
import os, random
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from pathlib import Path

import jpype
from jpype import JClass, getDefaultJVMPath

import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.corpus import stopwords
from functools import lru_cache

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import json


[nltk_data] Downloading package stopwords to C:\Users\Laptop
[nltk_data]     Dunyası\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to C:\Users\Laptop
[nltk_data]     Dunyası\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Laptop
[nltk_data]     Dunyası\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [11]:
df = pd.read_csv("Datasets/final_dengeli_yorumlar.csv", encoding="utf-8-sig")
print(df.shape)
print(df.head())

(25060, 2)
                                            yorumlar    duygu
0                 Fiyatına göre gayet güzel bir ürün  pozitif
1  sıfır kutusunda sorunsuz fiyatı ram 3gb olduğu...  pozitif
2  makinanın heryeri yapışkanlı gibi tuttuğum yer...     notr
3                şahane telefon satıcıya selam olsun  pozitif
4               kalitesiz bir ürün tavsiye etmiyorum  negatif


In [12]:
ZEMBEREK_PATH = "zemberek-full.jar"
JVM_PATH = r"C:\Users\Laptop Dunyası\Downloads\eclipse-java-2024-06-R-win32-x86_64\eclipse\plugins\org.eclipse.justj.openjdk.hotspot.jre.full.win32.x86_64_21.0.3.v20240426-1530\jre\bin\server\jvm.dll"  # Burayı kendi yoluna göre düzelt

if not jpype.isJVMStarted():
    jpype.startJVM(JVM_PATH, "-ea", f"-Djava.class.path={ZEMBEREK_PATH}")
else:
    print("JVM zaten çalışıyor.")

TurkishMorphology   = JClass("zemberek.morphology.TurkishMorphology")
TurkishTokenizer    = JClass("zemberek.tokenization.TurkishTokenizer")
TurkishSpellChecker = JClass("zemberek.normalization.TurkishSpellChecker")

morphology    = TurkishMorphology.createWithDefaults()
tokenizer     = TurkishTokenizer.DEFAULT
spell_checker = TurkishSpellChecker(morphology)

print("Zemberek hazır")

JVM zaten çalışıyor.
Zemberek hazır


In [ ]:
stop_words = set(stopwords.words('turkish'))

def clean_text(t: str) -> str:
    t = re.sub(r"<[^>]+>", " ", str(t))
    t = re.sub(r"http\S+|www\.\S+", " ", t)
    t = re.sub(r"[^0-9a-zçğıöşüA-ZÇĞİÖŞÜ\s]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t.lower()

_ALNUM_TR  = re.compile(r"^[0-9a-zçğıöşüA-ZÇĞİÖŞÜ]+$")
ASCII_ONLY = re.compile(r"^[a-z0-9]+$")

def zemberek_tokenize(text: str):
    toks = [str(x) for x in tokenizer.tokenizeToStrings(text)]
    return [w for w in toks if _ALNUM_TR.match(w)]

_DIACRITIC_MAP = {"c":"ç","g":"ğ","i":"ı","o":"ö","s":"ş","u":"ü"}

@lru_cache(maxsize=100_000)
def is_known(w: str) -> bool:
    ana = morphology.analyzeAndDisambiguate(w).bestAnalysis()
    return not (ana.isEmpty() or ana.get(0).isUnknown())

@lru_cache(maxsize=100_000)
def correct_token(w: str) -> str:
    w = w.lower()
    if len(w) <= 1:
        return w
    if spell_checker.check(w) or is_known(w):
        return w
    sugs = spell_checker.suggestForWord(w)
    if not sugs.isEmpty():
        cand = str(sugs.get(0)).lower()
        if spell_checker.check(cand) or is_known(cand):
            return cand
    if ASCII_ONLY.match(w):
        cand1 = {w[:i] + _DIACRITIC_MAP[ch] + w[i+1:] for i, ch in enumerate(w) if ch in _DIACRITIC_MAP}
        for c in cand1:
            if is_known(c): return c
        cand2 = {c[:i] + _DIACRITIC_MAP[ch] + c[i+1:] for c in cand1 for i, ch in enumerate(c) if ch in _DIACRITIC_MAP}
        for c in cand2:
            if is_known(c): return c
    return w

NEGATORS = {"değil","degil","yok","hiç","hic","asla"}

def add_neg_prefix(tokens, window=1):
    if not APPLY_NEGATION: return tokens
    out=[]; i=0
    while i < len(tokens):
        if tokens[i] in NEGATORS:
            out.append("not_"+tokens[i])
            if i+1 < len(tokens): out.append("not_"+tokens[i+1])
            i += 2
        else:
            out.append(tokens[i]); i += 1
    return out

EXTRA_STOPS = {
    "urun","ürün","urunu","ürünü","urunü","üründe",
    "orjinal","orijinal",
    "tesekkur","teşekkür","tesekkürler","teşekkürler",
    "gibi","falan","filan","vesaire","vs",
    "ayrica","ancak","lakin","zaten","yani"
}

stop_all = stop_words | EXTRA_STOPS

APPLY_NEGATION  = True
APPLY_STOPWORDS = True

def filter_stopwords(tokens):
    return [w for w in tokens if w not in stop_all] if APPLY_STOPWORDS else tokens

def preprocess_basic(text: str) -> str:
    s    = clean_text(text)
    toks = zemberek_tokenize(s)
    toks = [correct_token(t) for t in toks]
    toks = add_neg_prefix(toks, window=1)
    toks = filter_stopwords(toks)
    return " ".join(toks)
is_known.cache_clear()
correct_token.cache_clear()

In [14]:
pip install -U nlpaug transformers torch

Note: you may need to restart the kernel to use updated packages.


In [15]:
import nlpaug.augmenter.word as naw

ctx_sub = naw.ContextualWordEmbsAug(
    model_path="dbmdz/bert-base-turkish-cased",
    action="substitute",
    aug_p=0.10,
    device="cpu"
)

ctx_ins = naw.ContextualWordEmbsAug(
    model_path="dbmdz/bert-base-turkish-cased",
    action="insert",
    aug_p=0.08,
    device="cpu"
)

def augment_ctx_once(text, mode="sub"):
    if mode == "sub":
        return ctx_sub.augment(text)
    else:
        return ctx_ins.augment(text)

id2lbl = {0: "negatif", 1: "notr", 2: "pozitif"}

def model_label(text, _model=None, _tok=None):
    cls, _ = predict_text(text, _model, _tok)
    try:
        return id2lbl[int(cls)]
    except (ValueError, KeyError, TypeError):
        return str(cls).lower()

def augment_label_to_target(df, label, target_n, mode_cycle=("sub","ins"), checker=None, consistency_check=True, seed=42):
    rng = np.random.default_rng(seed)
    cur = df[df["duygu"] == label].copy()
    have = len(cur)
    need = max(0, target_n - have)
    if need == 0:
        return df.copy()

    src = cur.sample(n=min(have, need), replace=(have < need), random_state=seed)["yorumlar"].tolist()

    aug_texts = []
    m = len(mode_cycle)
    for i, s in enumerate(src):
        mode = mode_cycle[i % m]
        try:
            aug = augment_ctx_once(s, mode=mode)
            if isinstance(aug, list):
                aug = aug[0]
            aug_texts.append(aug)
        except Exception:
            continue

    df_aug = pd.DataFrame({"yorumlar": aug_texts[:need], "duygu": label})

    if consistency_check and (checker is not None) and not df_aug.empty:
            preds = df_aug["yorumlar"].apply(checker)
            df_aug = df_aug[preds.values == df_aug["duygu"].values]

    out = pd.concat([df, df_aug], ignore_index=True)
    out = out.drop_duplicates(subset=["yorumlar"]).reset_index(drop=True)
    return out

c:\Users\Laptop Dunyası\OneDrive\Masaüstü\sentiment_analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
MAX_VOCAB = 20000
MAX_LEN   = 100

In [17]:
df_bal = df.copy()

df_bal = augment_label_to_target(
    df_bal,
    label="notr",
    target_n=10_000,
    mode_cycle=("sub","ins"),
    consistency_check=True,
    seed=42
)

print("Sınıf sayımları (artırma sonrası):")
print(df_bal["duygu"].value_counts())

Sınıf sayımları (artırma sonrası):
duygu
pozitif    10000
negatif    10000
notr        9998
Name: count, dtype: int64


In [18]:
X_train, X_val, y_train, y_val = train_test_split(
    df_bal["yorumlar"], df_bal["duygu"],
    test_size=0.2,
    random_state=42,
    stratify=df_bal["duygu"]
)

train_df = pd.DataFrame({"yorumlar": X_train, "duygu": y_train})
val_df   = pd.DataFrame({"yorumlar": X_val,   "duygu": y_val})

In [19]:
tok = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tok.fit_on_texts(train_df["yorumlar"])

X_train_seq = tok.texts_to_sequences(train_df["yorumlar"])
X_val_seq   = tok.texts_to_sequences(val_df["yorumlar"])

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_val_pad   = pad_sequences(X_val_seq,   maxlen=MAX_LEN, padding="post", truncating="post")

In [20]:
le = LabelEncoder()
y_train_enc = le.fit_transform(train_df["duygu"])
y_val_enc   = le.transform(val_df["duygu"])

y_train_cat = to_categorical(y_train_enc)
y_val_cat   = to_categorical(y_val_enc)

In [ ]:
np.savez_compressed(
    "preprocessed_arrays.npz",
    X_train_pad=X_train_pad, y_train_cat=y_train_cat,
    X_val_pad=X_val_pad,     y_val_cat=y_val_cat
)

with open("tokenizer.json","w",encoding="utf-8") as f:
    f.write(tok.to_json())

import json
os.environ["PYTHONHASHSEED"] = "42"
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)
with open("label_encoder_classes.json","w",encoding="utf-8") as f:
    json.dump(le.classes_.tolist(), f, ensure_ascii=False)

In [22]:
df_bal.to_csv("artirilmis_temizlenmis.csv", index=False, encoding="utf-8-sig")